## 03 · MCP araçlarını kullanan ajan (Colab · T4 GPU)

`server.py` içindeki MCP sunucusu stdio üzerinden başlatılır, araç listesi modele fonksiyon tanımı olarak verilir. Model (**Qwen3-4B-Instruct-2507**) hangi aracı hangi parametreyle çağıracağına kendisi karar verir; araç sonuçları `role=tool` mesajı olarak geri beslenir.

> Çalışma zamanı: *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("job-opportunity-mcp-agent").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/job-opportunity-mcp-agent.git
if IN_COLAB and Path("job-opportunity-mcp-agent").exists():
    !git -C job-opportunity-mcp-agent pull -q
    %cd job-opportunity-mcp-agent
    !pip -q install -r requirements.txt transformers accelerate
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### MCP sunucusuna bağlanma
Jupyter'de `sys.stderr` gerçek bir dosya olmadığı için sunucu logları ayrı bir dosyaya yönlendiriliyor. Her hücrede bağlantı `async with` ile açılıp kapanıyor (anyio iptal kapsamı aynı görevde kapanmalı).

In [ ]:
from contextlib import asynccontextmanager
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["server.py"])

@asynccontextmanager
async def mcp_session():
    with open("mcp_server.log", "a") as errlog:
        async with stdio_client(SERVER, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                yield session, await session.initialize()

async with mcp_session() as (session, info):
    print(f"sunucu: {info.serverInfo.name} | protokol: {info.protocolVersion}")
    for t in (await session.list_tools()).tools:
        print(f"- {t.name}: {t.description.splitlines()[0]}")
        print("   parametreler:", list(t.inputSchema['properties']))

### Modeli yükleme

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
print(f"{MODEL_ID} yüklendi: {time.time() - t0:.0f} sn, GPU belleği {torch.cuda.memory_allocated() / 2**30:.1f} GB")

def generate(messages, tools):
    prompt = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    return text.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

### Ajanı çalıştırma

In [ ]:
import textwrap
from agent import run_agent

def show(result, width=150):
    for s in result.steps:
        if s.kind == "tool_call":
            print(f"🔧 {s.tool}({s.content})")
        elif s.kind == "tool_result":
            body = s.content if len(s.content) < 260 else s.content[:260] + " ..."
            print(textwrap.indent(textwrap.fill(body, width - 6), "   ↳ "))
        elif s.kind == "error":
            print(f"⚠️  {s.content}")
        else:
            print("\n💬 YANIT:\n" + textwrap.fill(s.content, width))

SORU = ("Uzaktan çalışabileceğim ve PostgreSQL kullanan ilanları bul; ilk ilanın ayrıntılarını getirip "
        "bana uygun olup olmadığını kısaca söyle. Python, Docker ve AWS biliyorum, Türkiye'den çalışıyorum.")

#### 1. sürüm · yalnız arama araçları (`search_postings`, `get_posting`)

In [ ]:
async with mcp_session() as (session, _):
    t0 = time.time()
    v1 = await run_agent(SORU, session, generate, max_steps=5, allowed_tools={"search_postings", "get_posting"})
show(v1)
print(f"\nsüre: {time.time() - t0:.1f} sn")

Model konum alanındaki bölge kısıtını (ör. *Remote US or Ontario, Canada*) gözden kaçırabiliyor: ilan "uzaktan" etiketli olduğu için Türkiye'den çalışan kullanıcıya uygun sayılabiliyor. Bu kararı modelin yorumuna bırakmak yerine kural tabanlı bir **`check_fit`** aracı ekledim; sistem istemi de öneri yapmadan önce bu aracın çağrılmasını istiyor.

#### 2. sürüm · kural tabanlı ön kontrol aracıyla (`check_fit`)

In [ ]:
async with mcp_session() as (session, _):
    t0 = time.time()
    v2 = await run_agent(SORU, session, generate, max_steps=6)
show(v2)
print(f"\nsüre: {time.time() - t0:.1f} sn")

### Hata durumu: olmayan ilan
Araç `ToolError` döndürdüğünde modelin bilgi uydurmadan durumu açıklaması bekleniyor.

In [ ]:
async with mcp_session() as (session, _):
    result = await run_agent("12345 numaralı ilanın maaş aralığı nedir?", session, generate)
show(result)